# Fig. 5: Detection of previously overlooked viral infections and flagging of non-genuine detections

Prepare viral annotation tables for the Fig. 5 analyses.
Run the companion scripts at the indicated steps.


In [1]:
import os
from collections import defaultdict
import pandas as pd
import numpy as np
from Bio import SeqIO

In [2]:
# Define paths
basedir = "/path/to/project"

# Output directory
path_out = f"{basedir}/output/revise_final/Fig5/"

# Input files
# BLASTx output directory
path_in = f"{basedir}/data/250819_4474samples/blastout_all_protein/"
# Contig FASTA files
path_in_fasta = f"{basedir}/data/250819_4474samples/contig_viral_hit/"
# Sample information
sradata_path = f"{basedir}/data/Infection_Prediction_Stacking_0_sample_list_TN1000.txt"
sra_metadata_dir = f"{basedir}/data/SRA_data/"
# Blacklisted viral families
blacklist_path = f"{basedir}/data/black_list_TN_ISG_low_new_format.txt"
# BLASTn results
path_in_blastn = f"{basedir}/data/250909_blastn/virus_hit_all_contigs_megablast.txt"

if not os.path.exists(path_out):
    os.makedirs(path_out)
print("saving files:", path_out)


saving files: /Users/coco/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/revise_final/Fig5/


In [3]:
def get_all_taxids(root_dir, takonkit_path):
    """
    Collect taxids from BLASTx top hits across samples.
    """
    # Initialize the taxid set
    taxids = set()

    # Read BLASTx hits from each sample
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if not (filename.endswith(".txt")):  # Skip non-result files
                continue

            # Path to the BLAST result file
            full_path = os.path.join(dirpath, filename)

            # Read BLAST hits
            col_name = [
                "qseqid", "sseqid", "pident", "length", "mismatch",
                "gapopen", "qstart", "qend", "sstart", "send", "evalue", 
                "bitscore", "qframe", "staxids", "stitle", "qlen", 
                "slen", "qcovhsp", "scovhsp"
            ]

            if os.path.getsize(full_path) == 0:
                print("file size 0 ")
                continue

            one_sample = pd.read_table(full_path, header=None, dtype={13:str})  # Read taxids as strings
            one_sample.columns = col_name

            # Check that every hit has a valid bit score
            if one_sample['bitscore'].isna().sum() > 0:
                print("NaN", filename)
            if (one_sample['bitscore'].dtype != np.float64) and (one_sample['bitscore'].dtype != int):
                print("dtype", filename, one_sample['bitscore'].dtype)
                print(one_sample['bitscore'])

            # Sort hits by bit score
            one_sample_sored = one_sample.sort_values("bitscore", ascending=False)

            # Remove hits without taxids
            one_sample_sored = one_sample_sored.dropna(subset=["staxids"])

            # Keep the top hit for each query
            one_sample_uniq = one_sample_sored.groupby("qseqid").first()

            # Collect taxids
            for line_taxid in one_sample_uniq["staxids"].to_list():
                if (line_taxid is None) or (line_taxid!=line_taxid):
                    continue
                raw_taxids = line_taxid.split(";")
                for i in range(len(raw_taxids)):
                    taxids.add(raw_taxids[i])

    # Write one taxid per line
    with open(takonkit_path, "w") as f:
        for taxid in sorted(list(taxids)):
            f.write(taxid + "\n")
    print(f"{len(taxids)} taxids in total")

    return taxids


def get_viral_hits(input_file, out_dir, lineage_info, blacklst):
    """
    Annotate top hits and retain non-phage, non-blacklisted viral contigs.
    """
    # Count hits spanning multiple taxonomic groups
    num_ambiguous_hit = 0

    # Assign the confusion-matrix group
    cm_group = input_file.split("/")[-2]

    # Output file path
    output_folder = out_dir + "/taxonomy_output/"
    if not(os.path.exists(output_folder)):
        os.mkdir(output_folder)

    output_all_path = output_folder + "/" + cm_group + "." + input_file.split("/")[-1].split(".")[0] + ".blastx.taxonomy.txt"
    output_viral_path = output_folder + "/" + cm_group + "." + input_file.split("/")[-1].split(".")[0] + ".blastx.virus.txt"
    output_viral_black_path = output_folder + "/" + cm_group + "." + input_file.split("/")[-1].split(".")[0] + ".blastx.virus.black.txt"

    # Read BLAST hits
    col_name = [
        "qseqid", "sseqid", "pident", "length", "mismatch",
        "gapopen", "qstart", "qend", "sstart", "send", "evalue", 
        "bitscore", "qframe", "staxids", "stitle", "qlen", "slen",
        "qcovhsp", "scovhsp"
    ]

    one_sample = pd.read_table(input_file, header=None, dtype={13:str})
    one_sample.columns = col_name

    # Sort hits by bit score
    one_sample_sored = one_sample.sort_values("bitscore", ascending=False)

    # Remove hits without taxids
    one_sample_sored = one_sample_sored.dropna(subset=["staxids"])

    # Keep the top hit for each query
    one_sample_uniq = one_sample_sored.groupby("qseqid").first()

    # Count all contigs
    num_all_hits = len(set(one_sample_uniq.index.to_list()))

    # Add taxonomic annotations
    one_sample_tax = one_sample_uniq.copy()
    one_sample_tax["staxids_lst"] = one_sample_tax["staxids"].apply(lambda x: x.split(";") if((x is not None) and (x==x)) else x)
    one_sample_tax["name_lst"] = one_sample_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "fgs"] for taxid in x] if((x is not None) and (x==x)) else x)
    one_sample_tax["group_lst"] = one_sample_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "domain"] for taxid in x] if((x is not None) and (x==x)) else x)

    # Check for mixed viral and non-viral taxonomic groups
    for idx in one_sample_tax.index:
        groups = one_sample_tax.loc[idx, "group_lst"]
        if ((groups is not None) and (groups == groups)) and any("Viruses" in item for item in groups) and (len(set(groups)) > 1):
            num_ambiguous_hit += 1

    one_sample_tax["taxid_blastx"] = one_sample_tax["staxids_lst"].apply(lambda x: x[0] if((x is not None) and (x==x)) else x)  
    one_sample_tax["family"] = one_sample_tax["staxids_lst"].apply(lambda x: lineage_info.loc[x[0], "family"] if((x is not None) and (x==x)) else x)
    one_sample_tax["genus"] = one_sample_tax["staxids_lst"].apply(lambda x: lineage_info.loc[x[0], "genus"] if((x is not None) and (x==x)) else x)
    one_sample_tax["species"] = one_sample_tax["staxids_lst"].apply(lambda x: lineage_info.loc[x[0], "species"] if((x is not None) and (x==x)) else x)

    # Mark hits as viral only when all taxonomic groups are viral
    one_sample_tax["is_virus"] = one_sample_tax["group_lst"].apply(lambda x: 1 if((x is not None) and (x==x)) and all("Viruses" in item for item in x) else 0)

    # Retain viral hits
    one_sample_virus = one_sample_tax.copy()
    one_sample_virus = one_sample_virus[one_sample_virus["is_virus"] == 1]

    # Count viral contigs
    num_viral_hits = len(set(one_sample_virus.index.to_list()))

    # Read blacklisted viral families
    blackvirus = []
    with open(blacklst, "r") as f_in:
        for line in f_in:
            blackvirus.append(line.strip())

    # Exclude blacklisted viral families
    one_sample_black_removed = one_sample_virus.copy()
    one_sample_black_removed = one_sample_black_removed[~one_sample_black_removed["family"].isin(blackvirus)]

    # Count viral contigs after blacklist filtering
    num_viral_hits_after_filter = len(set(one_sample_black_removed.index.to_list()))

    # Save results
    one_sample_tax.to_csv(output_all_path, sep="\t")
    one_sample_virus.to_csv(output_viral_path, sep="\t")
    one_sample_black_removed.to_csv(output_viral_black_path, sep="\t")

    return one_sample_black_removed, num_all_hits, num_viral_hits, num_viral_hits_after_filter, num_ambiguous_hit


def count_viral_contigs(path_in: str, path_out, lineage_info, blacklist_path, infection_th=1):
    """
    Collect viral contigs and summarize detection by confusion-matrix group.
    """
    # Initialize summaries for each confusion-matrix group
    cm_hits = defaultdict(list)
    cm_props = defaultdict(list)
    cm_class = defaultdict(list)
    sample_class = defaultdict(lambda: defaultdict(str))

    # Initialize the ambiguous-hit count and first-file flag
    num_amb_hit = 0
    is_fist_file = 1

    # Read all BLAST result files
    for dirpath, dirnames, filenames in os.walk(path_in):
        for filename in filenames:
            if not (filename.endswith(".txt")):  # Skip non-result files
                continue

            # Path to the BLAST result file
            full_path = os.path.join(dirpath, filename)

            if os.path.getsize(full_path) == 0:
                print("filesize 0")
                continue

            # Annotate and save viral contigs
            df_one_sample, num_all_contigs, num_viral_contigs, num_viral_black, num_amb_hit_each = get_viral_hits(full_path, path_out, lineage_info, blacklist_path)
            num_amb_hit += num_amb_hit_each

            # Store contig counts by confusion-matrix group
            cm_group = full_path.split('/')[-2]
            cm_hits[cm_group].append(num_viral_black)
            cm_props[cm_group].append(num_viral_black / num_all_contigs)
    
            # Classify samples using the contig-count threshold
            if num_viral_black >= infection_th:
                cm_class[cm_group].append("1")
            else:
                cm_class[cm_group].append("0")

            # Combine sample results
            if is_fist_file == 1:
                df_all = df_one_sample.copy()
                is_fist_file = 0
            else:
                df_all = pd.concat([df_all, df_one_sample])

            # Create the sample summary table
            sample = full_path.split('/')[-1].split(".")[0].split("_")[0]
            if num_viral_black >= infection_th:
                sample_class[sample]["viral_contig"] = "1"
            else:
                sample_class[sample]["viral_contig"] = "0"
            sample_class[sample]["cm"] = cm_group

    print("Virus ambiguous hit:", num_amb_hit)
    print("=======")

    for k, v in cm_hits.items():
        print(k, np.mean(v))
    for k, v in cm_props.items():
        print(k, f"{np.mean(v):.2}")

    print("=======")
    return cm_hits, cm_props, cm_class, sample_class, df_all


def collect_from_tsv(tsv_file, output_file):
    """
    Collect sequences listed in a TSV into a single multi-FASTA file.

    Parameters:
        tsv_file (str): Path to TSV file with columns [path, qseqid]
        output_file (str): Path to the output FASTA file
    """
    df = pd.read_table(tsv_file, sep="\t")
    with open(output_file, "w") as out_f:
        for fasta_file in sorted(list(set(df["path"].to_list()))):
            record_dict = SeqIO.index(fasta_file, "fasta")
            contigs = df.loc[df["path"] == fasta_file, "qseqid"].to_list()
            for contig in contigs:
                if contig in record_dict:
                    SeqIO.write(record_dict[contig], out_f, "fasta")
                else:
                    print(f"[Warning] {contig} not found in {fasta_file}")


def get_seq_from_fasta(fasta_file, contig_name):
    """
    Return the sequence of a named contig from a FASTA file.

    Parameters:
        fasta_file (str): Path to the FASTA file
        contig_name (str): Contig ID (matches the part after '>' up to first space)

    Returns:
        str: Sequence as a string, or None if not found
    """
    record_dict = SeqIO.index(fasta_file, "fasta")
    if contig_name in record_dict:
        return record_dict[contig_name].seq
    else:
        return None

# TaxonKit

In [4]:
# Collect taxids from all samples
taxids = get_all_taxids(path_in, path_out + "taxids.txt")

17083 taxids in total


Run `Fig5_table_taxonkit.sh` in the taxonkit environment.

# Identify phages

In [5]:
# Read TaxonKit output
taxinfo = pd.read_table(path_out + "taxids_long.txt", header=None, dtype={0:str})
taxinfo.columns = ["taxid", "lineage"]

# Split the lineage into taxonomic ranks
taxinfo["domain"] = taxinfo["lineage"].apply(lambda x: x.split(";")[0] if x == x else x)
taxinfo["kingdom"] = taxinfo["lineage"].apply(lambda x: x.split(";")[1] if x == x else x)
taxinfo["phylum"] = taxinfo["lineage"].apply(lambda x: x.split(";")[2] if x == x else x)
taxinfo["class"] = taxinfo["lineage"].apply(lambda x: x.split(";")[3] if x == x else x)
taxinfo["order"] = taxinfo["lineage"].apply(lambda x: x.split(";")[4] if x == x else x)
taxinfo["family"] = taxinfo["lineage"].apply(lambda x: x.split(";")[5] if x == x else x)
taxinfo["genus"] = taxinfo["lineage"].apply(lambda x: x.split(";")[6] if x == x else x)
taxinfo["species"] = taxinfo["lineage"].apply(lambda x: x.split(";")[7] if x == x else x)

# Check missing values
print("#NaN:", taxinfo.isna().sum().sum())
# Fill missing values
taxinfo = taxinfo.fillna("-")

# Check table dimensions
print("taxonkit output original size:", taxinfo.shape[0])

# Identify phage taxa
taxinfo["is_phage"] = 0
taxinfo["r1_Caudoviricetes"] = 0
taxinfo["r2_includes_phage"] = 0
taxinfo["r3_Inoviridae"] = 0
taxinfo["r4_Lavidaviridae"] = 0
taxinfo["r5_Microviridae"] = 0
phage_group_order = set()

for idx in taxinfo.index:
    # 1. Caudoviricetes
    if taxinfo.loc[idx, "class"] == "Caudoviricetes":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r1_Caudoviricetes"] = 1
    # 2. Unclassified viral species containing "phage"
    if ("phage" in taxinfo.loc[idx, "species"]) and (taxinfo.loc[idx, "domain"] == 'unclassified Viruses domain'):
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r2_includes_phage"] = 1
        phage_group_order.add(taxinfo.loc[idx, "order"])
    # 3. Inoviridae
    if taxinfo.loc[idx, "family"] == "Inoviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r3_Inoviridae"] = 1
    # 4. Lavidaviridae
    if taxinfo.loc[idx, "family"] == "Lavidaviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r4_Lavidaviridae"] = 1
    # 5. Microviridae
    if taxinfo.loc[idx, "family"] == "Microviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r5_Microviridae"] = 1

# Summarize phage orders
phage_group_order = sorted(list(phage_group_order))
print("phage order:", phage_group_order)

# Check table dimensions
print("Caudoviricetes:", taxinfo.loc[taxinfo["r1_Caudoviricetes"] == 1, :].shape[0])
print("phage:", taxinfo.loc[taxinfo["r2_includes_phage"] == 1, :].shape[0])
print(" * not Caudoviricetes:", taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :].shape[0])
print("Inoviridae:", taxinfo.loc[taxinfo["r3_Inoviridae"] == 1, :].shape[0])
print("Lavidaviridae:", taxinfo.loc[taxinfo["r4_Lavidaviridae"] == 1, :].shape[0])
print("Microviridae:", taxinfo.loc[taxinfo["r5_Microviridae"] == 1, :].shape[0])

# Save the annotation table
taxinfo.to_csv(path_out + "taxids_phage.tsv", sep="\t")

# Save the phage taxid list
phages = taxinfo.loc[taxinfo["is_phage"] == 1, "taxid"].to_list()
with open(path_out + "phage_list.txt", "w") as f:
    for pid in phages:
        f.write(pid + "\n")

# Preview results
taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :]

#NaN: 18
taxonkit output original size: 17083
phage order: ['Crassvirales', 'unclassified Caudoviricetes order', 'unclassified Viruses order']
Caudoviricetes: 676
phage: 306
 * not Caudoviricetes: 8
Inoviridae: 1
Lavidaviridae: 0
Microviridae: 1


,taxid,lineage,domain,kingdom,phylum,class,order,family,genus,species,is_phage,r1_Caudoviricetes,r2_includes_phage,r3_Inoviridae,r4_Lavidaviridae,r5_Microviridae
5937,1868660,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,uncultured Mediterranean phage,1,0,1,0,0,0
9572,2740180,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,Streptomyces phage Dagobah,1,0,1,0,0,0
9813,278008,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,uncultured phage,1,0,1,0,0,0
11061,2961709,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,Salmonella phage vB_Sal_PHB48,1,0,1,0,0,0
11477,3034680,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,Staphylococcus phage Southeast,1,0,1,0,0,0
11847,3093841,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,Bacillus phage CM1,1,0,1,0,0,0
12724,38018,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,Bacteriophage sp.,1,0,1,0,0,0
15498,707152,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,uncultured marine phage,1,0,1,0,0,0


# Add taxonomy

In [6]:
# Read TaxonKit output
taxid_in = path_out + "taxids_lineage.txt"
lineage_info = pd.read_table(taxid_in, header=None)
lineage_info.columns = ["taxid", "domain", "fgs"]

# Check missing values
print("#NaN beginning:", lineage_info.isna().sum().sum())
nan_rows =lineage_info[lineage_info.isna().any(axis=1)]
print(nan_rows)
print()

# Add family, genus and species columns
lineage_info["taxid"] = lineage_info["taxid"].astype(str)
lineage_info = lineage_info.set_index("taxid")
lineage_info["family"] = lineage_info["fgs"].apply(lambda x: x.split(";")[0] if x==x else np.nan)  
lineage_info["genus"] = lineage_info["fgs"].apply(lambda x: x.split(";")[1] if x==x else np.nan)  
lineage_info["species"] = lineage_info["fgs"].apply(lambda x: x.split(";")[2] if x==x else np.nan)  

# Label phage taxa as phage_group
phage_ids = []
with open(path_out + "phage_list.txt", "r") as f:
    for line in f:
        line = line.strip()
        phage_ids.append(line)

for idx in lineage_info.index:
    if idx in phage_ids:
        lineage_info.loc[idx, "domain"] = "phage_group"

# Check missing values
print("#NaN end:", lineage_info.isna().sum().sum())
nan_rows =lineage_info[lineage_info.isna().any(axis=1)]
print(nan_rows)
print()

# Fill missing values
lineage_info = lineage_info.fillna("-")

print(f"{lineage_info.shape[0]} taxids in total\n")

# Preview hits from each domain group
print("All domain:", set(lineage_info["domain"].to_list()), "\n")
for domain in set(lineage_info["domain"].to_list()):
    print(f"##### {domain} ####")
    print(f"size: {lineage_info[lineage_info["domain"] == domain].shape[0]}")
    print(lineage_info[["domain", "species"]][lineage_info["domain"] == domain].head(3))
    print()

# Preview results
lineage_info

#NaN beginning: 4
        taxid domain  fgs
1426  1230063    NaN  NaN
1429  1230068    NaN  NaN

#NaN end: 10
        domain  fgs family genus species
taxid                                   
1230063    NaN  NaN    NaN   NaN     NaN
1230068    NaN  NaN    NaN   NaN     NaN

17083 taxids in total

All domain: {'phage_group', 'unclassified Viruses domain', 'Bacteria', 'unclassified unclassified entries domain', '-', 'Eukaryota', 'Archaea', 'unclassified other entries domain'} 

##### phage_group ####
size: 686
              domain             species
taxid                                   
1029988  phage_group   Erskinevirus EaH2
103216   phage_group     Peduovirus Wphi
1051675  phage_group  Waedenswilvirus S6

##### unclassified Viruses domain ####
size: 5153
                              domain                         species
taxid                                                               
1000504  unclassified Viruses domain  Alphainfluenzavirus influenzae
1001142  unclassified V

,domain,fgs,family,genus,species
taxid,,,,,
100,Bacteria,Xanthobacteraceae;Ancylobacter;Ancylobacter aq...,Xanthobacteraceae,Ancylobacter,Ancylobacter aquaticus
100035,Eukaryota,unclassified Pleosporales family;Massariosphae...,unclassified Pleosporales family,Massariosphaeria,Massariosphaeria phaeospora
1000413,Eukaryota,Sapindaceae;Acer;Acer yangbiense,Sapindaceae,Acer,Acer yangbiense
1000504,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
1000590,Bacteria,Staphylococcaceae;Staphylococcus;Staphylococcu...,Staphylococcaceae,Staphylococcus,Staphylococcus epidermidis
...,...,...,...,...,...
999809,Eukaryota,Ustilaginaceae;Sporisorium;Sporisorium reilianum,Ustilaginaceae,Sporisorium,Sporisorium reilianum
999810,Eukaryota,Sclerotiniaceae;Botrytis;Botrytis cinerea,Sclerotiniaceae,Botrytis,Botrytis cinerea
999883,unclassified Viruses domain,Marseilleviridae;unclassified Marseilleviridae...,Marseilleviridae,unclassified Marseilleviridae genus,Lausannevirus


# Extract viral top hits

In [7]:
# Extract viral top hits
infection_threshod = 1
cm_hits, cm_props, cm_class, sample_class, df_all = count_viral_contigs(path_in, path_out, lineage_info, blacklist_path, infection_threshod)

# Create the viral contig table
df = df_all.copy().reset_index()
df["sample"] = df["qseqid"].apply(lambda x: x.split("_")[0])
# Add confusion-matrix labels
sampledf = pd.DataFrame(sample_class).T
df = df.merge(sampledf, left_on = "sample", right_index=True, how="left")

# Add viral contig counts per sample
virus_hit_count = pd.DataFrame(df.value_counts("sample"))
df = df.merge(virus_hit_count, on="sample", how="left")
df = df.sort_values("sample")
df = df.sort_values("count", ascending=False)

# Check missing values
print("#NaN:", df.isna().sum().sum())

# Save results
df.to_csv(path_out + "virus_all.txt", sep="\t")

Virus ambiguous hit: 2530
FP 0.8367911479944675
TP 10.783582089552239
FN 8.288686605981795
TN 0.384
FP 0.0014
TP 0.016
FN 0.013
TN 0.00078
#NaN: 0


# BLASTn

In [8]:
# Prepare BLASTn inputs
# Create a table of contig IDs and FASTA paths
df = pd.read_table(path_out + "virus_all.txt", index_col=0)
df = df.drop("viral_contig", axis=1)
df_fasta = df[["qseqid", "cm", "sample"]].copy()
for idx in df_fasta.index:
    df_fasta.loc[idx, "path"] = f"{path_in_fasta}{df_fasta.loc[idx, "cm"]}/{df_fasta.loc[idx, "sample"]}.viral_hit_contigs.fa"
df_fasta = df_fasta[["path", "qseqid"]]
# Save the annotation table
df_fasta.to_csv(path_out + "virus_contig_path.txt", sep="\t", index=False)
# Collect viral contigs into a FASTA file
collect_from_tsv(path_out + "virus_contig_path.txt", path_out + "virus_hit_all_contigs.fasta")

Run `Fig5_table_blastn.sh` on the HPC and download its output to `path_in_blastn` before continuing.

# TaxonKit for BLASTn hits

In [9]:
# Prepare TaxonKit inputs
# Taxid list for TaxonKit
takonkit_path_2 = path_out + "taxids_blastn.txt"

# Initialize the taxid set
taxids = set()

# Read BLASTn hits
col_name= [
    "qseqid", "sseqid", "pident", "length", 
    "mismatch", "gapopen", "qstart", "qend", 
    "sstart", "send", "evalue", "bitscore", 
    "qframe", "staxids", "stitle", "qlen", 
    "slen", "qcovs", "qcovhsp"
]

blastn = pd.read_table(path_in_blastn, header=None, dtype={13:str})  # Read taxids as strings
blastn.columns = col_name
print("blastn output original size:", blastn.shape)
print("#NaN:", blastn.isna().sum().sum())

# Sort hits by bit score
blastn_sorted = blastn.sort_values("bitscore", ascending=False)

# Remove hits without taxids
blastn_sorted = blastn_sorted.dropna(subset=["staxids"])

# Keep the top hit for each query
blastn_uniq = blastn_sorted.groupby("qseqid").first()
print("blastn output size after choosing tophits:", blastn_uniq.shape)

# Collect taxids
for line_taxid in blastn_uniq["staxids"].to_list():
    if (line_taxid is None) or (line_taxid!=line_taxid):
        continue
    raw_taxids = line_taxid.split(";")
    for i in range(len(raw_taxids)):
        taxids.add(raw_taxids[i])

# Write one taxid per line
with open(takonkit_path_2, "w") as f:
    for taxid in taxids:
        f.write(taxid + "\n")

print(f"{len(taxids)} taxids in total")

# Preview results
blastn_uniq.head(10)

blastn output original size: (2914726, 19)
#NaN: 0
blastn output size after choosing tophits: (11981, 18)
1035 taxids in total


,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,qframe,staxids,stitle,qlen,slen,qcovs,qcovhsp
qseqid,,,,,,,,,,,,,,,,,,
ERR10662039_124963,gi|2118777509|gb|MZ367372.1|,95.794,214,9,0,1,214,5756,5543,8.140000e-91,346.0,1,336959,Chicken astrovirus isolate CAV/Belgium/4134_00...,215,7480,99,99
ERR10662039_29322,gi|389618860|gb|JN582313.1|,94.014,852,51,0,1,852,1349,2200,0.000000e+00,1291.0,1,336959,Chicken astrovirus isolate VF06-7/3 capsid pro...,852,2229,100,100
ERR10662039_58854,gi|2118777509|gb|MZ367372.1|,95.690,464,20,0,1,464,5793,6256,0.000000e+00,747.0,1,336959,Chicken astrovirus isolate CAV/Belgium/4134_00...,464,7480,100,100
ERR10662039_61653,gi|2118777509|gb|MZ367372.1|,98.870,354,4,0,1,354,3854,3501,9.850000e-177,632.0,1,336959,Chicken astrovirus isolate CAV/Belgium/4134_00...,354,7480,100,100
ERR10662039_85530,gi|389618860|gb|JN582313.1|,96.438,365,13,0,1,365,178,542,8.040000e-168,603.0,1,336959,Chicken astrovirus isolate VF06-7/3 capsid pro...,367,2229,99,99
ERR11505140_111036,gi|2414813051|ref|XR_008221636.1|,98.565,209,3,0,1,209,4996,5204,4.670000e-98,370.0,1,564181,PREDICTED: Peromyscus californicus insignis un...,209,6015,100,100
ERR11505145_2536,gi|1908055260|ref|XR_004943307.1|,99.842,2528,4,0,1,2528,3396,5923,0.000000e+00,4647.0,1,38674,PREDICTED: Onychomys torridus uncharacterized ...,2613,5923,97,97
ERR11505147_9442,gi|1908055261|ref|XR_004943308.1|,98.251,1887,24,3,1,1880,5459,3575,0.000000e+00,3293.0,1,38674,PREDICTED: Onychomys torridus uncharacterized ...,1880,6013,100,100
ERR11505154_3122,gi|1908055259|ref|XR_004943306.1|,99.820,3883,7,0,1,3883,2065,5947,0.000000e+00,7132.0,1,38674,PREDICTED: Onychomys torridus uncharacterized ...,3883,5955,100,100


Run `Fig5_table_taxonkit_blastn.sh` in the taxonkit environment.

# Identify phages in BLASTn hits

In [10]:
# Identify phages
# Read TaxonKit output
taxinfo = pd.read_table(path_out + "taxids_long_blastn.txt", header=None, dtype={0:str})
taxinfo.columns = ["taxid", "lineage"]

# Split the lineage into taxonomic ranks
taxinfo["domain"] = taxinfo["lineage"].apply(lambda x: x.split(";")[0] if x == x else x)
taxinfo["kingdom"] = taxinfo["lineage"].apply(lambda x: x.split(";")[1] if x == x else x)
taxinfo["phylum"] = taxinfo["lineage"].apply(lambda x: x.split(";")[2] if x == x else x)
taxinfo["class"] = taxinfo["lineage"].apply(lambda x: x.split(";")[3] if x == x else x)
taxinfo["order"] = taxinfo["lineage"].apply(lambda x: x.split(";")[4] if x == x else x)
taxinfo["family"] = taxinfo["lineage"].apply(lambda x: x.split(";")[5] if x == x else x)
taxinfo["genus"] = taxinfo["lineage"].apply(lambda x: x.split(";")[6] if x == x else x)
taxinfo["species"] = taxinfo["lineage"].apply(lambda x: x.split(";")[7] if x == x else x)

# Check missing values
print("#NaN:", taxinfo.isna().sum().sum())
# Fill missing values
taxinfo = taxinfo.fillna("-")
# Check table dimensions
print("taxonkit result size:", taxinfo.shape[0])

# Identify phage taxa
taxinfo["is_phage"] = 0
taxinfo["r1_Caudoviricetes"] = 0
taxinfo["r2_includes_phage"] = 0
taxinfo["r3_Inoviridae"] = 0
taxinfo["r4_Lavidaviridae"] = 0
taxinfo["r5_Microviridae"] = 0
phage_group_order = set()

for idx in taxinfo.index:
    # 1. Caudoviricetes
    if taxinfo.loc[idx, "class"] == "Caudoviricetes":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r1_Caudoviricetes"] = 1
    # 2. Unclassified viral species containing "phage"
    if ("phage" in taxinfo.loc[idx, "species"]) and (taxinfo.loc[idx, "domain"] == 'unclassified Viruses domain'):
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r2_includes_phage"] = 1
        phage_group_order.add(taxinfo.loc[idx, "order"])
    # 3. Inoviridae
    if taxinfo.loc[idx, "family"] == "Inoviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r3_Inoviridae"] = 1
    # 4. Lavidaviridae
    if taxinfo.loc[idx, "family"] == "Lavidaviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r4_Lavidaviridae"] = 1
    # 5. Microviridae
    if taxinfo.loc[idx, "family"] == "Microviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r5_Microviridae"] = 1

# Summarize phage orders
phage_group_order = sorted(list(phage_group_order))
print("phage order:", phage_group_order)

# Check table dimensions
print("Caudoviricetes:", taxinfo.loc[taxinfo["r1_Caudoviricetes"] == 1, :].shape[0])
print("phage:", taxinfo.loc[taxinfo["r2_includes_phage"] == 1, :].shape[0])
print(" * not Caudoviricetes:", taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :].shape[0])
print("Inoviridae:", taxinfo.loc[taxinfo["r3_Inoviridae"] == 1, :].shape[0])
print("Lavidaviridae:", taxinfo.loc[taxinfo["r4_Lavidaviridae"] == 1, :].shape[0])
print("Microviridae:", taxinfo.loc[taxinfo["r5_Microviridae"] == 1, :].shape[0])

# Save the annotation table
taxinfo.to_csv(path_out + "taxids_phage_blastn.tsv", sep="\t")

# Save the phage taxid list
phages = taxinfo.loc[taxinfo["is_phage"] == 1, "taxid"].to_list()
with open(path_out + "phage_list_blastn.txt", "w") as f:
    for pid in phages:
        f.write(pid + "\n")

# Preview results
taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :]

#NaN: 9
taxonkit result size: 1035
phage order: []
Caudoviricetes: 0
phage: 0
 * not Caudoviricetes: 0
Inoviridae: 0
Lavidaviridae: 0
Microviridae: 1


,taxid,lineage,domain,kingdom,phylum,class,order,family,genus,species,is_phage,r1_Caudoviricetes,r2_includes_phage,r3_Inoviridae,r4_Lavidaviridae,r5_Microviridae


In [11]:
# Add taxonomic annotations to the taxid list
# Input file path
taxid_in = path_out + "taxids_lineage_blastn.txt"

# Read the lineage table
lineage_info = pd.read_table(taxid_in, header=None, dtype={0:str})
lineage_info.columns = ["taxid", "domain", "fgs"]

# Set the fallback taxonomy for taxid 3414891
lineage_info.loc[lineage_info["taxid"]=="3414891", "domain"] = "unclassified Viruses domain"
lineage_info.loc[lineage_info["taxid"]=="3414891", "fgs"] = "unclassified Bunyavirales;unclassified Bunyavirales;unclassified Bunyavirales"

# Check missing values
print("#NaN:", lineage_info.isna().sum().sum())

# Add family, genus and species columns
lineage_info = lineage_info.set_index("taxid")
lineage_info["family"] = lineage_info["fgs"].apply(lambda x: x.split(";")[0] if x==x else np.nan)
lineage_info["genus"] = lineage_info["fgs"].apply(lambda x: x.split(";")[1] if x==x else np.nan)
lineage_info["species"] = lineage_info["fgs"].apply(lambda x: x.split(";")[2] if x==x else np.nan)


# Label phage taxa as phage_group
phage_ids = []
with open(path_out + "phage_list_blastn.txt", "r") as f:
    for line in f:
        line = line.strip()
        phage_ids.append(line)

for idx in lineage_info.index:
    if idx in phage_ids:
        lineage_info.loc[idx, "domain"] = "phage_group"

# Fill missing values
lineage_info = lineage_info.fillna("-")

# Check table dimensions
print(f"{lineage_info.shape[0]} taxids in total\n")

# Preview hits from each domain group
print("All domain:", set(lineage_info["domain"].to_list()), "\n")
for domain in set(lineage_info["domain"].to_list()):
    print(f"##### {domain} ####")
    print(f"size: {lineage_info[lineage_info["domain"] == domain].shape[0]}")
    print(lineage_info[["domain", "species"]][lineage_info["domain"] == domain].head(3))
    print()

# Preview results
lineage_info

#NaN: 0
1035 taxids in total

All domain: {'phage_group', 'unclassified Viruses domain', 'Bacteria', 'Eukaryota', 'unclassified other entries domain'} 

##### phage_group ####
size: 1
              domain                   species
taxid                                         
2929290  phage_group  Dipodfec virus RodF1_123

##### unclassified Viruses domain ####
size: 968
                              domain                           species
taxid                                                                 
31591    unclassified Viruses domain  Rotavirus alphagastroenteritidis
1157338  unclassified Viruses domain                 Canine astrovirus
432075   unclassified Viruses domain    Alphainfluenzavirus influenzae

##### Bacteria ####
size: 1
           domain               species
taxid                                  
1850246  Bacteria  Urechidicola croceus

##### Eukaryota ####
size: 59
          domain            species
taxid                              
37598  Eukaryota  

,domain,fgs,family,genus,species
taxid,,,,,
31591,unclassified Viruses domain,Sedoreoviridae;Rotavirus;Rotavirus alphagastro...,Sedoreoviridae,Rotavirus,Rotavirus alphagastroenteritidis
1157338,unclassified Viruses domain,Astroviridae;Mamastrovirus;Canine astrovirus,Astroviridae,Mamastrovirus,Canine astrovirus
432075,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
1172196,unclassified Viruses domain,Caliciviridae;Bavovirus;Bavaria virus,Caliciviridae,Bavovirus,Bavaria virus
11277,unclassified Viruses domain,Rhabdoviridae;Vesiculovirus;Vesiculovirus indiana,Rhabdoviridae,Vesiculovirus,Vesiculovirus indiana
...,...,...,...,...,...
1534545,unclassified Viruses domain,Picornaviridae;Avisivirus;Avisivirus B,Picornaviridae,Avisivirus,Avisivirus B
2759407,unclassified Viruses domain,Parvoviridae;Chaphamaparvovirus;Chaphamaparvov...,Parvoviridae,Chaphamaparvovirus,Chaphamaparvovirus anseriform4
2976460,unclassified Viruses domain,Parvoviridae;Protoparvovirus;Protoparvovirus u...,Parvoviridae,Protoparvovirus,Protoparvovirus ungulate4


In [12]:
# Annotate BLASTn top hits
# Read BLASTn hits
blastn_tax = blastn_uniq.copy()
blastn_tax["staxids_lst"] = blastn_tax["staxids"].apply(lambda x: x.split(";") if((x is not None) and (x==x)) else x)
blastn_tax["name_lst"] = blastn_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "fgs"] for taxid in x] if((x is not None) and (x==x)) else x)
blastn_tax["group_lst"] = blastn_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "domain"] for taxid in x] if((x is not None) and (x==x)) else x)

# Check taxonomic assignments
num_ambiguous_hit = 0
for idx in blastn_uniq.index:
    groups = blastn_tax.loc[idx, "group_lst"]
    names = blastn_tax.loc[idx, "name_lst"]
    # Count hits spanning viral and non-viral taxonomic groups
    if ((groups is not None) and (groups == groups)) and any("Viruses" in item for item in groups) and (len(set(groups)) > 1):
        num_ambiguous_hit += 1
        print("#### Different root ####")
        print(groups)
        print(names)
        print()
print("Num ambiguou hits:", num_ambiguous_hit)

# Assign family, genus and species from the first taxid
blastn_tax["taxid"] = blastn_tax["staxids_lst"].apply(lambda x: x[0] if((x is not None) and (x==x)) else x)  
blastn_tax["family"] = blastn_tax["taxid"].apply(lambda x: lineage_info.loc[x, "family"] if((x is not None) and (x==x)) else x)
blastn_tax["genus"] = blastn_tax["taxid"].apply(lambda x: lineage_info.loc[x, "genus"] if((x is not None) and (x==x)) else x)
blastn_tax["species"] = blastn_tax["taxid"].apply(lambda x: lineage_info.loc[x, "species"] if((x is not None) and (x==x)) else x)

# Check missing values
print("#NaN:", blastn_tax.isna().sum().sum())

# Mark hits as viral only when all taxonomic groups are viral
blastn_tax["is_virus"] = blastn_tax["group_lst"].apply(lambda x: 1 if((x is not None) and (x==x)) and all("Viruses" in item for item in x) else 0)

blastn_tax.columns = blastn_tax.columns.map(lambda x: x + "_blastn")  # Add the _blastn suffix to annotation columns

# Save results
blastn_tax.to_csv(path_out + "blastn_result_top_hit.tsv", sep="\t")

# Preview results
print("Size after adding taxonomy:", blastn_tax.shape)
blastn_tax.head(10)

Num ambiguou hits: 0
#NaN: 0
Size after adding taxonomy: (11981, 26)


,sseqid_blastn,pident_blastn,length_blastn,mismatch_blastn,gapopen_blastn,qstart_blastn,qend_blastn,sstart_blastn,send_blastn,evalue_blastn,...,qcovs_blastn,qcovhsp_blastn,staxids_lst_blastn,name_lst_blastn,group_lst_blastn,taxid_blastn,family_blastn,genus_blastn,species_blastn,is_virus_blastn
qseqid,,,,,,,,,,,,,,,,,,,,,
ERR10662039_124963,gi|2118777509|gb|MZ367372.1|,95.794,214,9,0,1,214,5756,5543,8.140000e-91,...,99,99,[336959],[Astroviridae;Avastrovirus;Chicken astrovirus],[unclassified Viruses domain],336959,Astroviridae,Avastrovirus,Chicken astrovirus,1
ERR10662039_29322,gi|389618860|gb|JN582313.1|,94.014,852,51,0,1,852,1349,2200,0.000000e+00,...,100,100,[336959],[Astroviridae;Avastrovirus;Chicken astrovirus],[unclassified Viruses domain],336959,Astroviridae,Avastrovirus,Chicken astrovirus,1
ERR10662039_58854,gi|2118777509|gb|MZ367372.1|,95.690,464,20,0,1,464,5793,6256,0.000000e+00,...,100,100,[336959],[Astroviridae;Avastrovirus;Chicken astrovirus],[unclassified Viruses domain],336959,Astroviridae,Avastrovirus,Chicken astrovirus,1
ERR10662039_61653,gi|2118777509|gb|MZ367372.1|,98.870,354,4,0,1,354,3854,3501,9.850000e-177,...,100,100,[336959],[Astroviridae;Avastrovirus;Chicken astrovirus],[unclassified Viruses domain],336959,Astroviridae,Avastrovirus,Chicken astrovirus,1
ERR10662039_85530,gi|389618860|gb|JN582313.1|,96.438,365,13,0,1,365,178,542,8.040000e-168,...,99,99,[336959],[Astroviridae;Avastrovirus;Chicken astrovirus],[unclassified Viruses domain],336959,Astroviridae,Avastrovirus,Chicken astrovirus,1
ERR11505140_111036,gi|2414813051|ref|XR_008221636.1|,98.565,209,3,0,1,209,4996,5204,4.670000e-98,...,100,100,[564181],[Cricetidae;Peromyscus;Peromyscus californicus],[Eukaryota],564181,Cricetidae,Peromyscus,Peromyscus californicus,0
ERR11505145_2536,gi|1908055260|ref|XR_004943307.1|,99.842,2528,4,0,1,2528,3396,5923,0.000000e+00,...,97,97,[38674],[Cricetidae;Onychomys;Onychomys torridus],[Eukaryota],38674,Cricetidae,Onychomys,Onychomys torridus,0
ERR11505147_9442,gi|1908055261|ref|XR_004943308.1|,98.251,1887,24,3,1,1880,5459,3575,0.000000e+00,...,100,100,[38674],[Cricetidae;Onychomys;Onychomys torridus],[Eukaryota],38674,Cricetidae,Onychomys,Onychomys torridus,0
ERR11505154_3122,gi|1908055259|ref|XR_004943306.1|,99.820,3883,7,0,1,3883,2065,5947,0.000000e+00,...,100,100,[38674],[Cricetidae;Onychomys;Onychomys torridus],[Eukaryota],38674,Cricetidae,Onychomys,Onychomys torridus,0


# Save tables

In [13]:
# Merge BLASTx and BLASTn annotations
# Read BLASTx results
df_all = pd.read_table(path_out + "virus_all.txt", index_col=0)

# Read BLASTn hits
df_blastn = pd.read_table(path_out + "blastn_result_top_hit.tsv")
print("df_blastn shape:", df_blastn.shape)

# Merge BLASTx and BLASTn results
df_all_blastn = df_all.merge(df_blastn, on="qseqid", how="left")
print("df_all_blastn shape:", df_all_blastn.shape)
print("#NaN", df_all_blastn.isna().sum().sum())

# Add query sequences
df_all_blastn["query_seq"] = df_all_blastn.apply(lambda row: get_seq_from_fasta(f"{path_in_fasta}{row["cm"]}/{row["sample"]}.viral_hit_contigs.fa", row["qseqid"]), axis=1)

# Save results
df_all_blastn.to_csv(path_out + "virus_all.blastn.tsv", sep="\t")
# Preview results
df_all_blastn

df_blastn shape: (11981, 27)
df_all_blastn shape: (14353, 57)
#NaN 61672


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,...,qcovhsp_blastn,staxids_lst_blastn,name_lst_blastn,group_lst_blastn,taxid_blastn,family_blastn,genus_blastn,species_blastn,is_virus_blastn,query_seq
0,SRR8445393_281302,QMW68894.1,71.2,52,15,0,158,3,183,234,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"(A, C, C, C, G, A, A, A, T, T, T, A, T, A, G, ..."
1,SRR8445393_357285,UNY42121.1,51.9,54,23,1,163,11,1483,1536,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"(G, C, T, C, T, T, C, C, G, A, T, C, T, A, G, ..."
2,SRR8445393_136264,QZW33720.1,97.9,190,4,0,610,41,96,285,...,99.0,['2748378'],['unclassified Cressdnaviricota family;unclass...,['unclassified Viruses domain'],2748378.0,unclassified Cressdnaviricota family,unclassified Cressdnaviricota genus,Cressdnaviricota sp.,1.0,"(C, G, G, C, A, T, G, T, C, T, C, A, T, T, G, ..."
3,SRR8445393_141616,QTE03397.1,62.3,53,17,1,160,11,10,62,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"(C, C, A, C, G, C, A, G, T, A, G, T, T, A, A, ..."
4,SRR8445393_160709,QTE03459.1,49.2,61,26,1,190,8,88,143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"(C, T, C, T, T, C, C, G, A, T, C, T, T, G, G, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14348,SRR24422863_7911,YP_010087183.1,61.7,162,62,0,6,491,172,333,...,100.0,['564181'],['Cricetidae;Peromyscus;Peromyscus californicus'],['Eukaryota'],564181.0,Cricetidae,Peromyscus,Peromyscus californicus,0.0,"(C, A, C, A, G, A, G, C, T, G, G, C, A, G, A, ..."
14349,SRR24422864_12018,YP_010087183.1,64.7,133,47,0,1,399,201,333,...,100.0,['564181'],['Cricetidae;Peromyscus;Peromyscus californicus'],['Eukaryota'],564181.0,Cricetidae,Peromyscus,Peromyscus californicus,0.0,"(A, T, C, A, G, T, T, T, A, G, A, G, G, C, T, ..."
14350,SRR24422865_31476,YP_010087183.1,63.6,151,55,0,663,1115,183,333,...,100.0,['564181'],['Cricetidae;Peromyscus;Peromyscus californicus'],['Eukaryota'],564181.0,Cricetidae,Peromyscus,Peromyscus californicus,0.0,"(G, A, C, A, T, T, G, A, C, T, T, T, G, G, C, ..."
14351,SRR24422866_55522,YP_010087183.1,61.8,165,63,0,485,979,169,333,...,100.0,['564181'],['Cricetidae;Peromyscus;Peromyscus californicus'],['Eukaryota'],564181.0,Cricetidae,Peromyscus,Peromyscus californicus,0.0,"(T, G, C, A, T, G, G, A, A, T, T, T, T, G, A, ..."


# Add SRA metadata

In [14]:
# Read prediction results
ml_out = pd.read_table(sradata_path, dtype={"ID": str})

# Read sample and host-taxonomy metadata
sra_meta = pd.read_table(
    sra_metadata_dir + "Aves_Mam_mbio_metadata_ISG_logan_ML_250616_2.txt",
    dtype={"ID": str, "BioProject_ID": str, "tax_id": "Int64"},
)
host_ranks = ["kingdom", "phylum", "class", "order", "family", "genus", "species"]
sra_meta = sra_meta.rename(columns={rank: rank + "_host" for rank in host_ranks})

# Select one metadata row per BioProject
sra_meta2 = pd.read_table(
    sra_metadata_dir + "output_table3_with_infection2.txt",
    dtype={"Project_Id": "Int64", "Project_Acc": str},
)
sra_meta2 = sra_meta2.groupby("Project_Id").first().reset_index()
project_columns = ["Project_Id", "Project_Acc", "Project_Title", "Project_Description"]

# Read the saved BLAST results and query sequences
df_virus = pd.read_table(
    path_out + "virus_all.blastn.tsv", index_col=0,
    dtype={"sample": str, "taxid_blastx": "Int64", "taxid_blastn": "Int64", "is_virus_blastn": "Int64"},
).drop(columns=["viral_contig"])

# Join SRA, BioProject and prediction metadata by sample and project ID
df_metadata = df_virus.merge(
    sra_meta, left_on="sample", right_on="ID", how="left", validate="many_to_one",
)
df_metadata = df_metadata.merge(
    sra_meta2[project_columns], left_on="BioProject_ID", right_on="Project_Acc",
    how="left", validate="many_to_one",
)
df_metadata["Project_Id"] = df_metadata["Project_Id"].astype("Int64")
df_metadata = df_metadata.merge(
    ml_out[["ID", "Model", "True_Label", "Prediction_score", "Prediction_Label"]],
    on="ID", how="left", validate="many_to_one",
)

missing_samples = df_metadata.loc[df_metadata["ID"].isna(), "sample"].unique().tolist()
if missing_samples:
    raise ValueError(f"Run IDs missing from SRA metadata: {missing_samples}")

# Verify CM labels against the prediction sample list
expected_cm = df_metadata["ID"].map(ml_out.set_index("ID", verify_integrity=True)["CM"])
cm_mismatch = df_metadata["cm"].ne(expected_cm)
if cm_mismatch.any():
    mismatch_samples = df_metadata.loc[cm_mismatch, "sample"].unique().tolist()
    raise ValueError(f"CM groups differ from the prediction sample list: {mismatch_samples}")

print("Contigs:", len(df_metadata), "Runs:", df_metadata["ID"].nunique())
print("CM groups (runs):")
print(df_metadata[["ID", "cm"]].drop_duplicates()["cm"].value_counts())
print("Missing SRA metadata:")
print(df_metadata[["ID", "BioProject_ID", "Host_species", "tax_id", "cm", "Project_Acc"]].isna().sum())

# Report differences between infection and ground-truth labels
label_mismatch = df_metadata["Infection"].ne(df_metadata["True_Label"])
print("Infection != True_Label (contigs / runs):",
      int(label_mismatch.sum()), "/", df_metadata.loc[label_mismatch, "ID"].nunique())


Contigs: 14353 Runs: 1744
CM groups (runs):
cm
FN    680
TP    526
FP    464
TN     74
Name: count, dtype: int64
Missing SRA metadata:
ID               0
BioProject_ID    0
Host_species     0
tax_id           0
cm               0
Project_Acc      1
dtype: int64
Infection != True_Label (contigs / runs): 193 / 18


In [15]:
# Select the metadata columns in the required order
column_order = [
    "Project_Id", "Project_Acc", "Project_Title", "Project_Description",
    "ID", "Host_species", "BioProject_ID", "Infection",
    "species_host", "tax_id", "kingdom_host", "phylum_host", "class_host",
    "order_host", "family_host", "genus_host", "nucleotide_DNA",
    "nucleotide_RNA", "ISG_level", "Model", "True_Label", "Prediction_score", "Prediction_Label",
    "qseqid", "query_seq",
    "sseqid", "pident", "length", "mismatch", "gapopen", "qstart", "qend",
    "sstart", "send", "evalue", "bitscore", "qframe", "staxids", "stitle", "qlen", "slen",
    "staxids_lst", "name_lst", "group_lst", "taxid_blastx", "family", "genus", "species",
    "is_virus", "sample", "cm", "count", "qcovhsp", "scovhsp",
    "sseqid_blastn", "pident_blastn", "length_blastn", "mismatch_blastn", "gapopen_blastn",
    "qstart_blastn", "qend_blastn", "sstart_blastn", "send_blastn", "evalue_blastn",
    "bitscore_blastn", "qframe_blastn", "staxids_blastn", "stitle_blastn",
    "qlen_blastn", "slen_blastn", "qcovs_blastn", "qcovhsp_blastn",
    "staxids_lst_blastn", "name_lst_blastn", "group_lst_blastn", "taxid_blastn",
    "family_blastn", "genus_blastn", "species_blastn", "is_virus_blastn",
]
df_metadata = df_metadata[column_order]

metadata_output_path = os.path.join(path_out, "virus_all_4474.meta.txt")
df_metadata.to_csv(metadata_output_path, sep="\t")
print("Saved:", metadata_output_path)
print("Shape:", df_metadata.shape)
df_metadata.head()


Saved: /Users/coco/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/revise_final/Fig5/virus_all_4474.meta.txt
Shape: (14353, 80)


,Project_Id,Project_Acc,Project_Title,Project_Description,ID,Host_species,BioProject_ID,Infection,species_host,tax_id,...,qcovs_blastn,qcovhsp_blastn,staxids_lst_blastn,name_lst_blastn,group_lst_blastn,taxid_blastn,family_blastn,genus_blastn,species_blastn,is_virus_blastn
0,515305,PRJNA515305,Analysis of possible mechanisms behind functio...,Long-term survival and integration of neural p...,SRR8445393,Rattus_norvegicus,PRJNA515305,Positive,Rattus_norvegicus,10116,...,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>
1,515305,PRJNA515305,Analysis of possible mechanisms behind functio...,Long-term survival and integration of neural p...,SRR8445393,Rattus_norvegicus,PRJNA515305,Positive,Rattus_norvegicus,10116,...,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>
2,515305,PRJNA515305,Analysis of possible mechanisms behind functio...,Long-term survival and integration of neural p...,SRR8445393,Rattus_norvegicus,PRJNA515305,Positive,Rattus_norvegicus,10116,...,99.0,99.0,['2748378'],['unclassified Cressdnaviricota family;unclass...,['unclassified Viruses domain'],2748378,unclassified Cressdnaviricota family,unclassified Cressdnaviricota genus,Cressdnaviricota sp.,1
3,515305,PRJNA515305,Analysis of possible mechanisms behind functio...,Long-term survival and integration of neural p...,SRR8445393,Rattus_norvegicus,PRJNA515305,Positive,Rattus_norvegicus,10116,...,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>
4,515305,PRJNA515305,Analysis of possible mechanisms behind functio...,Long-term survival and integration of neural p...,SRR8445393,Rattus_norvegicus,PRJNA515305,Positive,Rattus_norvegicus,10116,...,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>
